## Cell 1 — Install required packages

Run this once in your environment. If you already installed these packages, you can skip this cell.

In [ ]:
# If you are running locally, uncomment and run this once.
# !pip install -U langgraph langchain langchain-groq python-dotenv pandas pydantic

## Cell 2 — Imports

This cell imports Python libraries, LangGraph, and ChatGroq.

In [1]:
import os
import json
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, TypedDict
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

## Cell 3 — Environment and model setup

This cell loads your Groq API key from `.env` or from your terminal environment.

Create a `.env` file in your project root like this:

```env
GROQ_API_KEY=your_key_here
```

Do not hard-code the API key inside the notebook.

In [17]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY is missing. Add it to a .env file or set it in your terminal environment."
    )

GENERATOR_MODEL = os.getenv("GENERATOR_MODEL", "llama-3.3-70b-versatile")

# Use a different model for the judge to reduce same-model leniency.
# If this model is not enabled in your Groq account, set JUDGE_MODEL in .env to another available Groq model.
JUDGE_MODEL = os.getenv("JUDGE_MODEL", "qwen/qwen3-32b")

# Keep max_tokens moderate to avoid Groq request-size/token errors.
generator_llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name=GENERATOR_MODEL,
    temperature=0.2,
    max_tokens=1400,
)

judge_llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name=JUDGE_MODEL,
    temperature=0.0,
    max_tokens=900,
)

reviser_llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name=GENERATOR_MODEL,
    temperature=0.2,
    max_tokens=1400,
)

print(f"Groq models initialized. Writer/Reviser: {GENERATOR_MODEL} | Judge: {JUDGE_MODEL}")


Groq models initialized. Writer/Reviser: llama-3.3-70b-versatile | Judge: qwen/qwen3-32b


## Cell 4 — Choose the payload and section

Change these values when you want to generate another bank or another report section.

For now, start with governance because it is the section we are debugging.

In [3]:
# Change this path to your local payload path.
PAYLOAD_PATH = Path("gen_data/payload_BANK01.json")

# If you are testing inside this ChatGPT environment, this fallback path may exist.
if not PAYLOAD_PATH.exists():
    fallback = Path("/payload_BANK01.json")
    if fallback.exists():
        PAYLOAD_PATH = fallback

REPORTING_YEAR = 2024
SECTION_NAME = "governance"

print("Payload path:", PAYLOAD_PATH)
print("Section:", SECTION_NAME)

Payload path: gen_data\payload_BANK01.json
Section: governance


## Cell 5 — Load the prepared payload

This loads the JSON output from your data preparation phase.

In [4]:
with open(PAYLOAD_PATH, "r", encoding="utf-8") as f:
    payload = json.load(f)

print("Top-level payload keys:")
for key in payload.keys():
    print("-", key)

Top-level payload keys:
- metadata
- reporting_kpis
- bank
- financial_summary
- scope1
- scope2
- scope3_travel
- financed_emissions
- financed_emissions_equity
- financed_emissions_sovereign
- targets
- governance
- board_minutes
- climate_scenarios
- climate_risk_register
- physical_risk_exposures
- carbon_credits
- internal_carbon_price
- value_chain_map


## Cell 6 — Utility functions for inspecting JSON

`print_json_tree` helps you understand the structure of the payload without printing the entire file.

In [5]:
def print_json_tree(obj: Any, indent: int = 0, max_depth: int = 3, max_items: int = 8) -> None:
    """Print a compact tree view of a nested JSON-like object."""
    prefix = "  " * indent

    if indent >= max_depth:
        print(prefix + "...")
        return

    if isinstance(obj, dict):
        for i, (k, v) in enumerate(obj.items()):
            if i >= max_items:
                print(prefix + f"... ({len(obj) - max_items} more keys)")
                break
            type_name = type(v).__name__
            extra = f" len={len(v)}" if isinstance(v, (list, dict, str)) else ""
            print(prefix + f"{k}: {type_name}{extra}")
            print_json_tree(v, indent + 1, max_depth, max_items)

    elif isinstance(obj, list):
        print(prefix + f"list[{len(obj)}]")
        for item in obj[:min(len(obj), 2)]:
            print_json_tree(item, indent + 1, max_depth, max_items)
        if len(obj) > 2:
            print(prefix + f"... ({len(obj) - 2} more items)")

    else:
        preview = str(obj)
        if len(preview) > 120:
            preview = preview[:120] + "..."
        print(prefix + preview)


print_json_tree(payload, max_depth=2)

metadata: dict len=6
  bank_id: str len=6
    ...
  reporting_year: int
    ...
  comparative_years: list len=2
    ...
  data_gaps: list len=3
    ...
  pcaf_methodology: dict len=5
    ...
  vehicles_correction: dict len=4
    ...
reporting_kpis: dict len=17
  scope1_2024_tco2e: float
    ...
  scope2_location_2024_tco2e: float
    ...
  scope2_market_2024_tco2e: float
    ...
  scope3_travel_2024_tco2e: float
    ...
  financed_emissions_2024_tco2e: float
    ...
  carbon_intensity_2024_tco2e_per_meur: float
    ...
  green_loans_pct_2024: float
    ...
  climate_capex_2024_meur: float
    ...
  ... (9 more keys)
bank: dict len=18
  bank_id: str len=6
    ...
  bank_name: str len=25
    ...
  archetype: str len=15
    ...
  country: str len=2
    ...
  total_assets_meur: int
    ...
  headcount: int
    ...
  established_year: int
    ...
  reporting_currency: str len=3
    ...
  ... (10 more keys)
financial_summary: list len=3
  list[3]
    ...
    ...
  ... (1 more items)
scope1: 

## Cell 7 — Robust JSON extraction from LLM output

Sometimes Groq returns JSON inside markdown fences like ```json. This helper extracts the JSON safely.

In [6]:
def extract_json_from_llm(text: str) -> Dict[str, Any]:
    """Extract a JSON object from an LLM response, including markdown-fenced JSON."""
    if text is None:
        raise ValueError("LLM response is empty.")

    raw = str(text).strip()

    # Remove markdown fences if present.
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", raw, flags=re.DOTALL | re.IGNORECASE)
    if fenced:
        raw = fenced.group(1).strip()

    # First try direct parse.
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    # Fallback: extract first {...} block.
    start = raw.find("{")
    end = raw.rfind("}")

    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"No JSON object found in LLM output:\n{raw[:1000]}")

    candidate = raw[start:end + 1]
    return json.loads(candidate)


def call_llm_text(llm: ChatGroq, system_prompt: str, user_prompt: str) -> str:
    """Call a chat model and return raw text."""
    response = llm.invoke([
        ("system", system_prompt),
        ("human", user_prompt),
    ])
    return response.content


def call_llm_json(llm: ChatGroq, system_prompt: str, user_prompt: str) -> Dict[str, Any]:
    """Call a chat model and parse the output as JSON."""
    raw = call_llm_text(llm, system_prompt, user_prompt)
    try:
        return extract_json_from_llm(raw)
    except Exception as e:
        return {
            "overall_score": 0,
            "evidence_support_score": 0,
            "ifrs_alignment_score": 0,
            "specificity_score": 0,
            "hallucination_risk": "high",
            "approved": False,
            "main_issues": ["Judge returned invalid JSON."],
            "required_fixes": ["Retry judging or inspect raw judge output."],
            "raw_output": raw,
            "parse_error": str(e),
        }

## Cell 8 — Section configurations

This defines what each section is supposed to cover. You can expand this as you add IFRS S1 sections later.

In [7]:
SECTION_CONFIGS = {
    "governance": {
        "title": "Governance",
        "purpose": "Explain governance processes, controls and procedures used to monitor, manage and oversee climate-related risks and opportunities.",
        "expected_content": [
            "Board oversight of climate-related risks and opportunities",
            "Management responsibility and delegation",
            "Climate skills and competencies",
            "Climate-related remuneration and incentives",
            "Board and committee decisions during the reporting year",
            "ERM, assurance and control integration",
        ],
    },
    "strategy": {
        "title": "Strategy",
        "purpose": "Explain climate-related risks and opportunities that could affect the business model, strategy, value chain and financial planning.",
        "expected_content": [
            "Current and anticipated climate-related risks and opportunities",
            "Effects on business model and value chain",
            "Strategic response and financial planning effects",
            "Use of climate scenarios where relevant",
        ],
    },
    "risk_management": {
        "title": "Risk Management",
        "purpose": "Explain how climate-related risks are identified, assessed, prioritized, monitored and integrated into overall risk management.",
        "expected_content": [
            "Risk identification and assessment process",
            "Physical and transition risk management",
            "Integration with ERM",
            "Monitoring and escalation",
        ],
    },
    "metrics_and_targets": {
        "title": "Metrics and Targets",
        "purpose": "Disclose metrics and targets used to measure and monitor climate-related risks and opportunities.",
        "expected_content": [
            "Scope 1 and Scope 2 emissions",
            "Scope 3 and financed emissions where available",
            "Targets and progress",
            "Internal carbon price and carbon credits where relevant",
        ],
    },
    "financed_emissions": {
        "title": "Financed Emissions",
        "purpose": "Explain financed emissions by asset class, methodology, PCAF data quality and portfolio exposure.",
        "expected_content": [
            "Financed emissions by asset class",
            "Attribution methodology",
            "Data quality score",
            "Portfolio concentration and limitations",
        ],
    },
    "climate_resilience": {
        "title": "Climate Resilience",
        "purpose": "Explain resilience of the strategy and business model under climate-related scenarios.",
        "expected_content": [
            "Scenario families and temperature pathways",
            "Physical and transition impacts",
            "Financial impact and risk mitigation",
            "Business model resilience",
        ],
    },
    "climate_transition_plan": {
        "title": "Climate Transition Plan",
        "purpose": "Explain transition plan targets, actions, progress and dependencies.",
        "expected_content": [
            "Targets and milestones",
            "Actions and progress",
            "Governance of transition plan",
            "Carbon credits and internal carbon price if used",
        ],
    },
    "internal_carbon_price": {
        "title": "Internal Carbon Price",
        "purpose": "Explain whether and how an internal carbon price is applied.",
        "expected_content": [
            "Price level and type",
            "Application scope",
            "Benchmark reference",
            "Whether it applies to financed emissions",
        ],
    },
    "carbon_credits": {
        "title": "Carbon Credits",
        "purpose": "Explain use of carbon credits, credit quality, volumes and whether credits are purchased or retired.",
        "expected_content": [
            "Purchased and retired volumes",
            "Credit standards and types",
            "Additionality and permanence",
            "Relationship with operational emissions",
        ],
    },
}

section_config = SECTION_CONFIGS[SECTION_NAME]
section_config

{'title': 'Governance',
 'purpose': 'Explain governance processes, controls and procedures used to monitor, manage and oversee climate-related risks and opportunities.',
 'expected_content': ['Board oversight of climate-related risks and opportunities',
  'Management responsibility and delegation',
  'Climate skills and competencies',
  'Climate-related remuneration and incentives',
  'Board and committee decisions during the reporting year',
  'ERM, assurance and control integration']}

## Cell 9 — Year filtering helpers

Most sections should only use the selected reporting year. Governance and transition sections keep multiple years because trends are useful.

In [8]:
TREND_SECTIONS = {
    "governance",
    "metrics_and_targets",
    "transition_plan",
    "climate_transition_plan",
}


def should_keep_all_years(section_name: str) -> bool:
    return section_name in TREND_SECTIONS


def filter_records_by_year(value: Any, reporting_year: int = 2024) -> Any:
    """Filter nested JSON records to the reporting year where a year field exists."""

    year_columns = ["reporting_year", "year", "assessment_year", "fiscal_year"]

    if isinstance(value, list):
        filtered = []
        for item in value:
            if not isinstance(item, dict):
                filtered.append(item)
                continue

            found_year = None
            for col in year_columns:
                if col in item:
                    found_year = item[col]
                    break

            if found_year is None:
                filtered.append(item)
            else:
                try:
                    if int(found_year) == int(reporting_year):
                        filtered.append(item)
                except Exception:
                    filtered.append(item)
        return filtered

    if isinstance(value, dict):
        return {k: filter_records_by_year(v, reporting_year) for k, v in value.items()}

    return value

## Cell 10 — Compact governance evidence

This is important. It keeps governance metrics for all years, but only sends the most relevant board decisions to the LLM.

This prevents Groq request-size errors while keeping strong evidence.

In [9]:
def compact_governance_evidence(payload: Dict[str, Any], reporting_year: int = 2024) -> Dict[str, Any]:
    """Build a compact governance evidence package for the LLM."""

    governance = payload.get("governance", [])
    board_minutes = payload.get("board_minutes", [])
    bank = payload.get("bank", {})
    metadata = payload.get("metadata", {})
    reporting_kpis = payload.get("reporting_kpis", {})
    climate_risk_register = payload.get("climate_risk_register", [])

    governance_records = governance if isinstance(governance, list) else [governance]

    current_year_minutes = []
    if isinstance(board_minutes, list):
        for m in board_minutes:
            if not isinstance(m, dict):
                continue

            meeting_date = str(m.get("meeting_date", ""))
            meeting_year = meeting_date[:4]

            if meeting_year == str(reporting_year) or str(m.get("reporting_year", "")) == str(reporting_year):
                current_year_minutes.append(m)

    priority_words = [
        "approved",
        "endorsed",
        "reviewed",
        "transition",
        "scenario",
        "climate",
        "risk",
        "carbon",
        "esg",
        "target",
        "remuneration",
        "assurance",
    ]

    def minute_score(m: Dict[str, Any]) -> int:
        text = " ".join([
            str(m.get("agenda_topic", "")),
            str(m.get("decision_summary", "")),
            str(m.get("meeting_type", "")),
            str(m.get("committee_name", "")),
        ]).lower()
        return sum(1 for word in priority_words if word in text)

    current_year_minutes = sorted(current_year_minutes, key=minute_score, reverse=True)

    selected_minutes = []
    seen_decisions = set()

    for m in current_year_minutes:
        decision = str(m.get("decision_summary", "")).strip()
        if not decision or decision.lower() in {"nan", "none"}:
            continue

        # Avoid repeated identical synthetic decisions.
        normalized = re.sub(r"\s+", " ", decision.lower())
        if normalized in seen_decisions:
            continue
        seen_decisions.add(normalized)

        selected_minutes.append({
            "meeting_id": m.get("meeting_id"),
            "meeting_date": m.get("meeting_date"),
            "meeting_type": m.get("meeting_type"),
            "committee_name": m.get("committee_name"),
            "agenda_topic": m.get("agenda_topic"),
            "decision_summary": m.get("decision_summary"),
        })

        if len(selected_minutes) >= 8:
            break

    selected_risks = []
    if isinstance(climate_risk_register, list):
        for r in climate_risk_register[:10]:
            if isinstance(r, dict):
                selected_risks.append({
                    "risk_id": r.get("risk_id"),
                    "risk_category": r.get("risk_category"),
                    "risk_description": r.get("risk_description"),
                    "risk_level": r.get("risk_level"),
                    "mitigation_action": r.get("mitigation_action"),
                })

    return {
        "metadata": {
            "reporting_year": metadata.get("reporting_year", reporting_year),
            "comparative_years": metadata.get("comparative_years"),
        },
        "bank": bank,
        "governance_records_all_years": governance_records,
        "selected_board_decisions_2024": selected_minutes,
        "selected_climate_risks": selected_risks,
        "reporting_kpis": reporting_kpis,
        "evidence_note": (
            "Board minutes were compacted to the most relevant unique climate-related decisions "
            "to avoid sending excessive raw evidence to the LLM."
        ),
    }

## Cell 11 — Extract relevant evidence for each section

This cell decides which payload keys to send for each report section.

In [10]:
def extract_relevant_evidence(
    payload: Dict[str, Any],
    section_name: str,
    reporting_year: int = 2024,
) -> Dict[str, Any]:
    """Extract section-specific evidence from the prepared payload."""

    if section_name == "governance":
        return {
            "section_name": section_name,
            "reporting_year": reporting_year,
            **compact_governance_evidence(payload, reporting_year),
        }

    section_key_map = {
        "strategy": [
            "metadata", "bank", "financial_summary", "reporting_kpis", "targets",
            "climate_scenarios", "climate_risk_register", "physical_risk_exposures",
            "value_chain_map", "internal_carbon_price", "carbon_credits",
        ],
        "risk_management": [
            "metadata", "bank", "governance", "board_minutes", "climate_risk_register",
            "physical_risk_exposures", "climate_scenarios", "value_chain_map",
        ],
        "metrics_and_targets": [
            "metadata", "bank", "reporting_kpis", "financial_summary", "scope1", "scope2",
            "scope3_travel", "financed_emissions", "financed_emissions_equity",
            "financed_emissions_sovereign", "targets", "internal_carbon_price", "carbon_credits",
        ],
        "financed_emissions": [
            "metadata", "bank", "reporting_kpis", "financed_emissions",
            "financed_emissions_equity", "financed_emissions_sovereign",
            "financial_summary", "targets",
        ],
        "climate_resilience": [
            "metadata", "bank", "financial_summary", "climate_scenarios",
            "climate_risk_register", "physical_risk_exposures", "value_chain_map",
        ],
        "climate_transition_plan": [
            "metadata", "bank", "reporting_kpis", "financial_summary", "targets",
            "governance", "board_minutes", "climate_scenarios",
            "internal_carbon_price", "carbon_credits",
        ],
        "internal_carbon_price": [
            "metadata", "bank", "reporting_kpis", "internal_carbon_price", "targets",
        ],
        "carbon_credits": [
            "metadata", "bank", "reporting_kpis", "carbon_credits", "scope1", "scope2",
        ],
    }

    keys_to_extract = section_key_map.get(section_name, list(payload.keys()))

    evidence = {
        "section_name": section_name,
        "reporting_year": reporting_year,
    }

    for key in keys_to_extract:
        if key in payload:
            if should_keep_all_years(section_name):
                evidence[key] = payload[key]
            else:
                evidence[key] = filter_records_by_year(payload[key], reporting_year)

    return evidence


section_evidence = extract_relevant_evidence(payload, SECTION_NAME, REPORTING_YEAR)

print("Extracted evidence keys:")
for key in section_evidence.keys():
    print("-", key)

print("Approx evidence tokens:", len(json.dumps(section_evidence, ensure_ascii=False)) // 4)
print_json_tree(section_evidence, max_depth=3)

Extracted evidence keys:
- section_name
- reporting_year
- metadata
- bank
- governance_records_all_years
- selected_board_decisions_2024
- selected_climate_risks
- reporting_kpis
- evidence_note
Approx evidence tokens: 2141
section_name: str len=10
  governance
reporting_year: int
  2024
metadata: dict len=2
  reporting_year: int
    2024
  comparative_years: list len=2
    list[2]
      ...
      ...
bank: dict len=18
  bank_id: str len=6
    BANK01
  bank_name: str len=25
    Eurolux Universal Bank AG
  archetype: str len=15
    large_universal
  country: str len=2
    DE
  total_assets_meur: int
    850000
  headcount: int
    48000
  established_year: int
    1872
  reporting_currency: str len=3
    EUR
  ... (10 more keys)
governance_records_all_years: list len=3
  list[3]
    bank_id: str len=6
      ...
    reporting_year: int
      ...
    board_size: int
      ...
    independent_directors_pct: float
      ...
    esg_committee_exists: bool
      ...
    esg_committee_meeting

## Cell 12 — Prompt builders

This cell builds the generator and judge prompts.

The governance prompt forces the exact subsection structure.

In [11]:
GENERATOR_SYSTEM_PROMPT = """
You are a senior sustainability reporting specialist for a commercial bank.
You write concise, evidence-based IFRS S1/S2 style disclosure sections.
Use only the provided evidence. Do not invent facts, years, values, names, committees, assurance scopes, or decisions.
Avoid marketing language and vague claims.
Return markdown section text only.
""".strip()

JUDGE_SYSTEM_PROMPT = """
You are a strict ESG reporting quality reviewer with IFRS S1/S2, banking, audit, and PCAF knowledge.
You must judge whether the generated section is supported by the prepared evidence and aligned with the section requirements.

SCORING RULES — follow these exactly:
- 9-10: Reserved for near-perfect sections. Every subsection is present, every claim is evidenced, year-on-year trends are cited, board decisions are specific and varied, no hedging language, no limitation disclaimers.
- 7-8: Good section with minor gaps. All required subsections present but one or two content requirements are thin or missing specific data.
- 5-6: Structural or content issues. Missing subsections, or present but vague. Trends missing despite available data. Board decisions generic or repeated.
- 3-4: Multiple missing subsections or significant hallucination risk. Evidence poorly used.
- 1-2: Does not meet minimum disclosure requirements.

A score of 9 or 10 requires ALL of the following to be true:
- All required subsections are present with substantive content
- At least 3 distinct board/committee decisions are cited with dates
- Year-on-year trends are present for at least 3 governance metrics
- climate_on_board_agenda_pct is correctly interpreted as meeting frequency, not agenda time
- No limitation disclaimer appears
- No hedging language when data is clearly available
- Remuneration percentages are explicitly cited
- Assurance level is explained, not just named

If ANY of the above is false, score cannot exceed 8.
Be strict. Do not reward fluent writing if evidence use is weak.
Return valid JSON only.
""".strip()

REVISER_SYSTEM_PROMPT = """
You are a senior ESG disclosure editor.
Revise the section using only the provided evidence and the judge's required fixes.
Do not add unsupported claims. Do not include methodology notes or limitation disclaimers unless explicitly requested.
Return markdown section text only.
""".strip()


def build_section_generation_prompt(section_name: str, section_config: Dict[str, Any], evidence: Dict[str, Any]) -> str:
    governance_extra = ""

    if section_name == "governance":
        governance_extra = """
GOVERNANCE-SPECIFIC REQUIREMENTS:

You must structure the section using exactly these markdown subsections:

### Governance

#### Board oversight
Explain the role of the Full Board and board-level committee oversight.
Use evidence on board size, board independence, climate_on_board_agenda_pct, board reporting frequency, and specific board-level climate decisions.

#### Management responsibility
Explain management's role separately from board oversight.
Use evidence on management_committee_name, erm_integration_flag, major_transactions_climate_check, skills_development_programme, operational implementation and escalation.

#### Climate skills and competencies
Explain how the bank addresses climate-related skills and expertise.
Use year-on-year trend evidence for board_climate_expertise_pct where available.

#### Remuneration and incentives
Explain how climate or ESG performance is linked to remuneration.
Use exact percentages for ceo_esg_compensation_pct and executive_climate_compensation_pct.
Include year-on-year trends where available.

#### Board and committee decisions during the year
Cite at least 3 specific decisions from board_minutes evidence.
Use meeting date + committee or meeting type + decision summary.
Do not list too many repeated decisions. Select the strongest examples.

#### Assurance and controls
Explain assurance, internal controls, and ERM integration.
Use evidence on external_assurance_provider, assurance_scope, assurance_standard, and ERM integration.

Additional rules:
- Interpret climate_on_board_agenda_pct as the percentage of board meetings where climate appeared on the agenda, not agenda time.
- Include year-on-year comparisons where multi-year evidence is available.
- Do not add a generic limitation disclaimer at the end.
- Do not use vague phrases like “demonstrates commitment” unless supported by concrete evidence.
- Do not merge board oversight and management responsibility.
- If committee names vary across years, describe this neutrally as a management-level sustainability governance forum recorded under different names, not as instability.
"""

    return f"""
Generate the IFRS S2 report section below.

SECTION:
- Section key: {section_name}
- Section title: {section_config['title']}
- Purpose: {section_config['purpose']}

EXPECTED CONTENT:
{json.dumps(section_config['expected_content'], indent=2, ensure_ascii=False)}

{governance_extra}

PREPARED EVIDENCE:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

Return the report-ready section text only.
""".strip()


def build_judge_prompt(
    section_name: str,
    section_config: Dict[str, Any],
    evidence: Dict[str, Any],
    generated_text: str,
) -> str:
    governance_checks = ""
    verification_block = ""

    if section_name == "governance":
        governance_checks = """
STRICT GOVERNANCE JUDGE CHECKS:

The section must contain exactly these markdown subsections:
- ### Governance
- #### Board oversight
- #### Management responsibility
- #### Climate skills and competencies
- #### Remuneration and incentives
- #### Board and committee decisions during the year
- #### Assurance and controls

Reject or heavily penalize the section if:
1. Board oversight and management responsibility are mixed together.
2. Management responsibility is missing or vague.
3. The section does not use management_committee_name, ERM integration, major transaction climate checks, or skills programme evidence when available.
4. climate_on_board_agenda_pct is interpreted as agenda time instead of percentage of meetings with climate on the agenda.
5. Year-on-year trends are missing when multi-year governance data is available.
6. Fewer than 3 specific board or committee decisions are cited from board_minutes.
7. Remuneration linkage is missing despite compensation evidence being available.
8. The section ends with a generic limitation disclaimer.
9. The section uses vague language without evidence.
10. The section fails to discuss strategic decision-making, risk management integration, or trade-offs where evidence allows.

Scoring ceilings:
- If the required subsection structure is missing, overall_score cannot exceed 6.
- If management responsibility is absent, ifrs_alignment_score cannot exceed 6.
- If board minutes are not used, evidence_support_score cannot exceed 7.
- If climate_on_board_agenda_pct is misinterpreted, hallucination_risk must be "medium" or "high".
- If year-on-year trends are missing despite available evidence, overall_score cannot exceed 7.
- approved must be false if any mandatory subsection is missing.
"""

        verification_block = """
BEFORE SCORING — answer each question internally and then return the answers in verification_answers:
1. Are ALL required subsections present? (yes/no)
2. Are at least 3 distinct board decisions cited with dates? (yes/no — list them)
3. Are year-on-year trends present for expertise, remuneration, and meeting frequency? (yes/no)
4. Is climate_on_board_agenda_pct correctly interpreted as meeting frequency? (yes/no)
5. Is any limitation disclaimer present? (yes/no)
6. Are remuneration percentages explicitly cited? (yes/no)
7. Is assurance level explained beyond just naming provider and standard? (yes/no)

Count how many answers are "no".
- 0 "no" answers: score can reach 9-10
- 1 "no" answer: score cannot exceed 8
- 2 "no" answers: score cannot exceed 7
- 3+ "no" answers: score cannot exceed 6

Apply this ceiling strictly before assigning your final score.
"""

    return f"""
Evaluate the generated IFRS S2 section.

SECTION:
- Section key: {section_name}
- Section title: {section_config['title']}
- Purpose: {section_config['purpose']}

EXPECTED CONTENT:
{json.dumps(section_config['expected_content'], indent=2, ensure_ascii=False)}

{governance_checks}

PREPARED EVIDENCE:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

GENERATED SECTION:
{generated_text}

{verification_block}

Return JSON with exactly these keys:
{{
  "overall_score": 0,
  "evidence_support_score": 0,
  "ifrs_alignment_score": 0,
  "specificity_score": 0,
  "hallucination_risk": "low",
  "approved": false,
  "main_issues": [],
  "required_fixes": [],
  "verification_answers": {{
    "all_subsections_present": true,
    "distinct_decisions_cited": [],
    "yoy_trends_present": true,
    "agenda_pct_correctly_interpreted": true,
    "limitation_disclaimer_present": false,
    "remuneration_cited": true,
    "assurance_explained": true,
    "no_answers_count": 0
  }}
}}
""".strip()


def build_revision_prompt(
    section_name: str,
    section_config: Dict[str, Any],
    evidence: Dict[str, Any],
    previous_text: str,
    judge_result: Dict[str, Any],
) -> str:
    return f"""
Revise the IFRS S2 report section based on the judge result.

SECTION:
- Section key: {section_name}
- Section title: {section_config['title']}
- Purpose: {section_config['purpose']}

EXPECTED CONTENT:
{json.dumps(section_config['expected_content'], indent=2, ensure_ascii=False)}

JUDGE RESULT:
{json.dumps(judge_result, indent=2, ensure_ascii=False)}

PREPARED EVIDENCE:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

PREVIOUS SECTION:
{previous_text}

Return a corrected, report-ready markdown section only.
""".strip()


## Cell 13 — Rule-based governance structure check

This deterministic checker catches missing governance subsections before relying on the LLM judge.

In [12]:
def rule_check_governance_structure(section_text: str) -> Dict[str, Any]:
    """Deterministic governance structure check."""

    required_headings = [
        "### Governance",
        "#### Board oversight",
        "#### Management responsibility",
        "#### Climate skills and competencies",
        "#### Remuneration and incentives",
        "#### Board and committee decisions during the year",
        "#### Assurance and controls",
    ]

    text_lower = section_text.lower()
    missing = [heading for heading in required_headings if heading.lower() not in text_lower]

    return {
        "passed": len(missing) == 0,
        "missing_headings": missing,
    }


def rule_check_governance_content(section_text: str) -> Dict[str, Any]:
    """Extra deterministic governance content checks."""

    text = section_text.lower()

    checks = {
        "mentions_management": "management" in text,
        "mentions_erm": "erm" in text or "enterprise risk management" in text,
        "mentions_remuneration": "remuneration" in text or "compensation" in text,
        "mentions_assurance": "assurance" in text,
        "mentions_board_decision_language": any(word in text for word in ["approved", "endorsed", "reviewed"]),
        "possible_agenda_time_misinterpretation": "agenda time" in text or "dedicated to climate" in text,
        "has_limitation_disclaimer": any(
            phrase in text
            for phrase in [
                "we acknowledge this limitation",
                "absence of prepared evidence",
                "unable to provide",
                "evidence is limited",
                "will strive to provide",
            ]
        ),
    }

    failed = [k for k, v in checks.items() if v is False]
    hard_fail = checks["possible_agenda_time_misinterpretation"] or checks["has_limitation_disclaimer"]

    return {
        "checks": checks,
        "failed_soft_checks": failed,
        "hard_fail": hard_fail,
    }

## Cell 14 — LangGraph state and agents

This cell defines the graph state and the generator, judge, and reviser agents.

In [13]:
class ReviewState(TypedDict, total=False):
    section_name: str
    section_config: Dict[str, Any]
    evidence: Dict[str, Any]
    draft_text: str
    revised_text: str
    final_text: str
    judge_result: Dict[str, Any]
    revision_count: int
    max_revisions: int
    status: str


def apply_judge_hard_gates(judge_result: Dict[str, Any]) -> Dict[str, Any]:
    """Apply deterministic approval gates after the LLM judge returns JSON."""
    if not isinstance(judge_result, dict):
        judge_result = {}

    judge_result.setdefault("overall_score", 0)
    judge_result.setdefault("approved", False)
    judge_result.setdefault("required_fixes", [])
    judge_result.setdefault("main_issues", [])

    # Hard override: score below 7 cannot be approved.
    if float(judge_result.get("overall_score", 0) or 0) < 7:
        judge_result["approved"] = False

    # Hard override: too many failed verification checks cannot be approved.
    verification = judge_result.get("verification_answers", {}) or {}
    no_count = int(verification.get("no_answers_count", 0) or 0)

    if no_count >= 2:
        judge_result["approved"] = False
        if "Section failed verification checks — see verification_answers for details" not in judge_result["required_fixes"]:
            judge_result["required_fixes"].append(
                "Section failed verification checks — see verification_answers for details"
            )

    # Score ceiling from verification checklist.
    if no_count == 1 and float(judge_result.get("overall_score", 0) or 0) > 8:
        judge_result["overall_score"] = 8
    elif no_count == 2 and float(judge_result.get("overall_score", 0) or 0) > 7:
        judge_result["overall_score"] = 7
    elif no_count >= 3 and float(judge_result.get("overall_score", 0) or 0) > 6:
        judge_result["overall_score"] = 6

    return judge_result


def generator_agent(state: ReviewState) -> ReviewState:
    prompt = build_section_generation_prompt(
        section_name=state["section_name"],
        section_config=state["section_config"],
        evidence=state["evidence"],
    )

    draft = call_llm_text(
        llm=generator_llm,
        system_prompt=GENERATOR_SYSTEM_PROMPT,
        user_prompt=prompt,
    )

    state["draft_text"] = draft.strip()
    state["revision_count"] = 0
    state["status"] = "drafted"
    return state


def judge_agent(state: ReviewState) -> ReviewState:
    text_to_judge = state.get("revised_text") or state.get("draft_text") or ""

    if state["section_name"] == "governance":
        structure_check = rule_check_governance_structure(text_to_judge)
        content_check = rule_check_governance_content(text_to_judge)

        if not structure_check["passed"]:
            state["judge_result"] = apply_judge_hard_gates({
                "overall_score": 5,
                "evidence_support_score": 6,
                "ifrs_alignment_score": 5,
                "specificity_score": 5,
                "hallucination_risk": "medium",
                "approved": False,
                "main_issues": ["Required governance subsection structure is missing."],
                "required_fixes": [f"Add missing headings: {structure_check['missing_headings']}"],
                "verification_answers": {
                    "all_subsections_present": False,
                    "distinct_decisions_cited": [],
                    "yoy_trends_present": False,
                    "agenda_pct_correctly_interpreted": True,
                    "limitation_disclaimer_present": False,
                    "remuneration_cited": False,
                    "assurance_explained": False,
                    "no_answers_count": 4,
                },
                "rule_check": structure_check,
            })
            state["status"] = "rejected_by_rule_check"
            return state

        if content_check["hard_fail"]:
            state["judge_result"] = apply_judge_hard_gates({
                "overall_score": 6,
                "evidence_support_score": 6,
                "ifrs_alignment_score": 6,
                "specificity_score": 6,
                "hallucination_risk": "medium",
                "approved": False,
                "main_issues": ["Governance content failed deterministic quality checks."],
                "required_fixes": [
                    "Remove agenda-time misinterpretation or limitation disclaimer.",
                    "Use only evidence-supported report language.",
                ],
                "verification_answers": {
                    "all_subsections_present": True,
                    "distinct_decisions_cited": [],
                    "yoy_trends_present": False,
                    "agenda_pct_correctly_interpreted": not content_check["checks"].get("possible_agenda_time_misinterpretation", False),
                    "limitation_disclaimer_present": content_check["checks"].get("has_limitation_disclaimer", False),
                    "remuneration_cited": content_check["checks"].get("mentions_remuneration", False),
                    "assurance_explained": content_check["checks"].get("mentions_assurance", False),
                    "no_answers_count": 3,
                },
                "rule_check": content_check,
            })
            state["status"] = "rejected_by_content_rule_check"
            return state

    judge_prompt = build_judge_prompt(
        section_name=state["section_name"],
        section_config=state["section_config"],
        evidence=state["evidence"],
        generated_text=text_to_judge,
    )

    judge_result = call_llm_json(
        llm=judge_llm,
        system_prompt=JUDGE_SYSTEM_PROMPT,
        user_prompt=judge_prompt,
    )

    judge_result = apply_judge_hard_gates(judge_result)

    state["judge_result"] = judge_result
    state["status"] = "judged"
    return state


def reviser_agent(state: ReviewState) -> ReviewState:
    previous_text = state.get("revised_text") or state.get("draft_text") or ""
    judge_result = state.get("judge_result", {})

    prompt = build_revision_prompt(
        section_name=state["section_name"],
        section_config=state["section_config"],
        evidence=state["evidence"],
        previous_text=previous_text,
        judge_result=judge_result,
    )

    revised = call_llm_text(
        llm=reviser_llm,
        system_prompt=REVISER_SYSTEM_PROMPT,
        user_prompt=prompt,
    )

    state["revised_text"] = revised.strip()
    state["revision_count"] = int(state.get("revision_count", 0)) + 1
    state["status"] = "revised"
    return state


def finalize_agent(state: ReviewState) -> ReviewState:
    state["final_text"] = state.get("revised_text") or state.get("draft_text") or ""
    state["status"] = "finalized"
    return state


## Cell 15 — Routing logic

The graph decides whether to revise again or finalize.

In [14]:
def route_after_judge(state: ReviewState) -> str:
    judge_result = state.get("judge_result", {})
    approved = bool(judge_result.get("approved", False))
    revision_count = int(state.get("revision_count", 0))
    max_revisions = int(state.get("max_revisions", 1))

    if approved:
        return "finalize"

    if revision_count >= max_revisions:
        return "finalize"

    return "revise"

## Cell 16 — Build the LangGraph workflow

This creates the graph: generator → judge → revise if needed → judge again → finalize.

In [15]:
builder = StateGraph(ReviewState)

builder.add_node("generator", generator_agent)
builder.add_node("judge", judge_agent)
builder.add_node("reviser", reviser_agent)
builder.add_node("finalize", finalize_agent)

builder.add_edge(START, "generator")
builder.add_edge("generator", "judge")

builder.add_conditional_edges(
    "judge",
    route_after_judge,
    {
        "revise": "reviser",
        "finalize": "finalize",
    },
)

builder.add_edge("reviser", "judge")
builder.add_edge("finalize", END)

graph = builder.compile()

print("LangGraph workflow compiled.")

LangGraph workflow compiled.


## Cell 17 — Run one section

This runs the workflow for the selected section.

The default now uses `max_revisions = 2`, so the judge can reject a weak draft and force up to two targeted revision cycles before finalization.


In [18]:
initial_state: ReviewState = {
    "section_name": SECTION_NAME,
    "section_config": section_config,
    "evidence": section_evidence,
    "max_revisions": 2,
}

result = graph.invoke(initial_state)

print("STATUS:", result.get("status"))
print("JUDGE RESULT:")
print(json.dumps(result.get("judge_result", {}), indent=2, ensure_ascii=False))

print("FINAL SECTION:")
print(result.get("final_text", ""))


APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `qwen/qwen3-32b` in organization `org_01kghdq29fe5wsk9080khp46mh` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 6072, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

## Cell 18 — Manual evaluation checklist

Even with an LLM judge, manually check the output before using it.

In [ ]:
def manual_checklist(section_name: str, final_text: str, judge_result: Dict[str, Any]) -> None:
    print("Manual checklist for:", section_name)
    print("- Did the section interpret all metrics correctly?")
    print("- Did it use the strongest available evidence?")
    print("- Did it avoid generic claims and marketing language?")
    print("- Did it avoid unsupported facts?")
    print("- Did it separate report narrative from data-quality limitations?")

    if section_name == "governance":
        print("- Did it separate board oversight from management responsibility?")
        print("- Did it explain management role under IFRS S2 governance requirements?")
        print("- Did it cite at least 3 board or committee decisions?")
        print("- Did it include year-on-year governance trends?")
        print("- Did it avoid interpreting agenda percentage as agenda time?")

    print("\nJudge approved:", judge_result.get("approved"))
    print("Overall score:", judge_result.get("overall_score"))


manual_checklist(SECTION_NAME, result.get("final_text", ""), result.get("judge_result", {}))


## Cell 19 — Save output files

This saves the section as markdown and saves the judge result as JSON.

In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

bank_id = payload.get("bank", {}).get("bank_id") or payload.get("metadata", {}).get("bank_id") or PAYLOAD_PATH.stem
section_slug = SECTION_NAME.lower().replace(" ", "_")

section_path = OUTPUT_DIR / f"{bank_id}_{section_slug}_section.md"
judge_path = OUTPUT_DIR / f"{bank_id}_{section_slug}_judge.json"

section_path.write_text(result.get("final_text", ""), encoding="utf-8")
judge_path.write_text(json.dumps(result.get("judge_result", {}), indent=2, ensure_ascii=False), encoding="utf-8")

print("Saved section:", section_path)
print("Saved judge result:", judge_path)

## Cell 20 — Run multiple sections one by one

Use this only after you are comfortable with the single-section workflow.

Tip: do not start with all sections. Run one section, evaluate it, then move to the next.

In [ ]:
def run_section(section_name: str, reporting_year: int = 2024, max_revisions: int = 2) -> Dict[str, Any]:
    if section_name not in SECTION_CONFIGS:
        raise ValueError(f"Unknown section: {section_name}. Available: {list(SECTION_CONFIGS.keys())}")

    evidence = extract_relevant_evidence(payload, section_name, reporting_year)

    state: ReviewState = {
        "section_name": section_name,
        "section_config": SECTION_CONFIGS[section_name],
        "evidence": evidence,
        "max_revisions": max_revisions,
    }

    return graph.invoke(state)


# Example usage:
# risk_result = run_section("risk_management", REPORTING_YEAR, max_revisions=2)
# print(risk_result["final_text"])
# print(json.dumps(risk_result["judge_result"], indent=2, ensure_ascii=False))


## Cell 21 — Optional: generate selected sections and save them

Uncomment the section list when you are ready.

In [ ]:
# sections_to_generate = [
#     "governance",
#     "risk_management",
#     "strategy",
#     "climate_resilience",
#     "metrics_and_targets",
#     "financed_emissions",
# ]
#
# all_results = {}
#
# for sec in sections_to_generate:
#     print("\n" + "=" * 80)
#     print("Generating:", sec)
#     sec_result = run_section(sec, REPORTING_YEAR, max_revisions=2)
#     all_results[sec] = sec_result
#
#     sec_path = OUTPUT_DIR / f"{bank_id}_{sec}_section.md"
#     judge_path = OUTPUT_DIR / f"{bank_id}_{sec}_judge.json"
#
#     sec_path.write_text(sec_result.get("final_text", ""), encoding="utf-8")
#     judge_path.write_text(json.dumps(sec_result.get("judge_result", {}), indent=2, ensure_ascii=False), encoding="utf-8")
#
#     print("Approved:", sec_result.get("judge_result", {}).get("approved"))
#     print("Score:", sec_result.get("judge_result", {}).get("overall_score"))


## Cell 22 — Optional: combine generated markdown sections into one report draft

This does not replace final review. It only combines saved section files.

In [ ]:
def combine_saved_sections(output_dir: Path, bank_id: str, section_order: List[str]) -> str:
    parts = []

    for sec in section_order:
        path = output_dir / f"{bank_id}_{sec}_section.md"
        if path.exists():
            parts.append(path.read_text(encoding="utf-8"))
        else:
            print("Missing section file:", path)

    return "\n\n".join(parts)


# Example usage:
# section_order = ["governance", "strategy", "risk_management", "metrics_and_targets"]
# report_markdown = combine_saved_sections(OUTPUT_DIR, bank_id, section_order)
# report_path = OUTPUT_DIR / f"{bank_id}_ifrs_s2_report_draft.md"
# report_path.write_text(report_markdown, encoding="utf-8")
# print("Saved report draft:", report_path)
